# 01 Data Understanding

## Goal

The goal of this notebook is to understand the structure of the raw Steam dataset before performing cleaning, feature engineering, EDA or modeling.

At this stage, we want to answer:

- What raw files are available?
- How many rows and columns does each file contain?
- What columns are available in each dataset?
- Are there obvious missing values?
- Which tables are useful for engagement and churn analysis?

## Load Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Define Paths

In [2]:
PROJECT_ROOT = Path("..")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "archive" / "steam"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

## Discover Raw Files

In [3]:
raw_files = list(RAW_DATA_PATH.glob("*"))
raw_files

[WindowsPath('../data/raw/archive/steam/achievements.csv'),
 WindowsPath('../data/raw/archive/steam/friends.csv'),
 WindowsPath('../data/raw/archive/steam/games.csv'),
 WindowsPath('../data/raw/archive/steam/history.csv'),
 WindowsPath('../data/raw/archive/steam/players.csv'),
 WindowsPath('../data/raw/archive/steam/prices.csv'),
 WindowsPath('../data/raw/archive/steam/private_steamids.csv'),
 WindowsPath('../data/raw/archive/steam/purchased_games.csv'),
 WindowsPath('../data/raw/archive/steam/reviews.csv')]

## Load Raw Datasets

In [4]:
csv_files = list(RAW_DATA_PATH.glob("*.csv"))
csv_files

[WindowsPath('../data/raw/archive/steam/achievements.csv'),
 WindowsPath('../data/raw/archive/steam/friends.csv'),
 WindowsPath('../data/raw/archive/steam/games.csv'),
 WindowsPath('../data/raw/archive/steam/history.csv'),
 WindowsPath('../data/raw/archive/steam/players.csv'),
 WindowsPath('../data/raw/archive/steam/prices.csv'),
 WindowsPath('../data/raw/archive/steam/private_steamids.csv'),
 WindowsPath('../data/raw/archive/steam/purchased_games.csv'),
 WindowsPath('../data/raw/archive/steam/reviews.csv')]

In [5]:
datasets = {}

for f in csv_files:
    dataset_name = f.stem
    datasets[dataset_name] = pd.read_csv(f)

## Dataset Overview

In [9]:
overview = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    total_missing = df.isna().sum().sum()

    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "total_missing_values": total_missing,
        "missing_values_pct": round((total_missing / total_cells) * 100, 2),
        "duplicated_rows": df.duplicated().sum()
    })

overview_df = pd.DataFrame(overview)
overview_df.to_csv(
    PROCESSED_DATA_PATH / "dataset_overview.csv",
    index=False
)
overview_df

,dataset,rows,cols,total_missing_values,missing_values_pct,duplicated_rows
0,achievements,1939027,4,176683,2.28,0
1,friends,424683,2,85222,10.03,0
2,games,98248,7,22558,3.28,0
3,history,10693879,3,0,0.00,0
4,players,424683,3,225537,17.70,0
5,prices,4414273,7,5444682,17.62,0
6,private_steamids,227963,1,0,0.00,0
7,purchased_games,102548,2,55607,27.11,0
8,reviews,1204534,8,2097,0.02,0


In [8]:
missing_report = []

for name, df in datasets.items():
    missing_counts = df.isna().sum()
    missing_pct = (df.isna().mean() * 100).round(2)

    for column in df.columns:
        missing_report.append({
            "dataset": name,
            "column": column,
            "missing_values": missing_counts[column],
            "missing_pct": missing_pct[column]
        })

missing_report_df = pd.DataFrame(missing_report)
missing_report_df.to_csv(
    PROCESSED_DATA_PATH / "missing_values_report.csv",
    index=False
)
missing_report_df

,dataset,column,missing_values,missing_pct
0,achievements,achievementid,0,0.00
1,achievements,gameid,0,0.00
2,achievements,title,4471,0.23
3,achievements,description,172212,8.88
4,friends,playerid,0,0.00
5,friends,friends,85222,20.07
6,games,gameid,0,0.00
7,games,title,3,0.00
8,games,developers,5559,5.66
9,games,publishers,5941,6.05


## Initial Observations

Based on the initial dataset overview:

- The dataset contains multiple files describing players, achievements, game activity, purchased games and social connections.
- The `history` table appears to be the main source of player activity because it contains achievement unlock events over time.
- The `purchased_games` table can be used to calculate how many games each player owns.
- The `friends` table can be used to calculate the size of each player's social network.
- The final analytical dataset should probably be built at player level, where each row represents one player.